# Sensor Failure Decision System V2

This notebook moves from sensor signals to inspection decisions using reproducible steps, measured tradeoffs, and explicit heuristic labels where data learning is not used.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)

RANDOM_STATE = 42
sns.set_style("whitegrid")


## Run Configuration

Single source of truth for dataset path and target. Column names stay aligned with source data.


In [ ]:
DATA_PATH = "/kaggle/input/datasets/patimejia/machine-failure-sensor-data-kaggle-source-stefan/data.csv"
TARGET = "fail"
TEST_SIZE = 0.20
VAL_SIZE = 0.20  # share of training split

INSPECTION_COST_PER_MACHINE = 1.0
FAILURE_LOSS_PER_MISSED_MACHINE = 20.0
BASE_DAILY_CAPACITY = 10


## Data Load and Structure

Data load checks shape, schema, and target distribution without noisy full-table dumps.


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\nTarget distribution:")
print(df[TARGET].value_counts().sort_index())

display(df.head(5))


## Data Quality Check

Nulls, duplicate rows, and numeric typing checks support a clean modeling path.


In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
})

duplicates = int(df.duplicated().sum())

display(quality)
print("Duplicate rows:", duplicates)


## Modeling Dataset

Feature matrix and target vector are defined once. Numeric preprocessing only; no categorical branch is added.


In [ ]:
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

numeric_features = X.columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        )
    ],
    remainder="drop",
)

print("Features used:", numeric_features)


## Split Plan

Train, validation, and test partitions preserve class ratio through stratification.


In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VAL_SIZE,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Class balance in full data:")
print(y.value_counts(normalize=True).sort_index().round(3))


## Model Benchmarks

Three baseline models are fit on the same preprocessing stack and scored on validation set.


In [ ]:
models = {
    "Logistic": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "Forest": RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

rows = []
fitted = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[("prep", preprocess), ("model", clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)

    rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_val, pred),
            "recall": recall_score(y_val, pred),
            "precision": precision_score(y_val, pred),
            "f1": f1_score(y_val, pred),
        }
    )
    fitted[name] = pipe

val_results = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
display(val_results)


## Validation Threshold Selection

Probability threshold is tuned on validation predictions from the top model. Decision point is selected by maximum F1. This is a metric-driven cutoff, not a business policy yet.


In [ ]:
best_model_name = val_results.loc[0, "model"]
best_pipe = fitted[best_model_name]

val_proba = best_pipe.predict_proba(X_val)[:, 1]
prec, rec, thresh = precision_recall_curve(y_val, val_proba)
f1_vals = (2 * prec * rec) / (prec + rec + 1e-12)

curve_df = pd.DataFrame({
    "threshold": np.append(thresh, 1.0),
    "precision": prec,
    "recall": rec,
    "f1": f1_vals,
})

best_idx = curve_df["f1"].idxmax()
best_threshold = float(curve_df.loc[best_idx, "threshold"])

print("Selected model:", best_model_name)
print("Best validation threshold:", round(best_threshold, 6))
display(curve_df.loc[best_idx, ["threshold", "precision", "recall", "f1"]])

plt.figure(figsize=(8, 5))
plt.plot(curve_df["threshold"], curve_df["precision"], label="precision")
plt.plot(curve_df["threshold"], curve_df["recall"], label="recall")
plt.plot(curve_df["threshold"], curve_df["f1"], label="f1")
plt.axvline(best_threshold, color="black", linestyle="--", label="selected")
plt.title("Validation Threshold Tradeoff")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.show()


## Final Test Evaluation

Final metrics are computed once on the untouched test split with the locked model and threshold.


In [ ]:
test_proba = best_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

print("Classification report on test set")
print(classification_report(y_test, test_pred, digits=3))

cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="viridis")
plt.title("Confusion Matrix (Test)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

error_table = pd.DataFrame(
    {
        "actual": y_test.values,
        "pred": test_pred,
        "failure_probability": test_proba,
    },
    index=y_test.index,
)
error_table["error_type"] = np.select(
    [
        (error_table.actual == 1) & (error_table.pred == 0),
        (error_table.actual == 0) & (error_table.pred == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)

display(error_table[error_table["error_type"] != "correct"].sort_values("failure_probability", ascending=False).head(15))


## Risk Queue and Daily Capacity

Risk queue ranks machines by predicted failure probability. Daily capacity determines how many get inspected today.


In [ ]:
risk_df = pd.DataFrame(
    {
        "actual": y_test.values,
        "failure_probability": test_proba,
    },
    index=y_test.index,
).sort_values("failure_probability", ascending=False)

risk_df["priority_rank"] = np.arange(1, len(risk_df) + 1)
risk_df["inspect_today"] = (risk_df["priority_rank"] <= BASE_DAILY_CAPACITY).astype(int)
risk_df["high_risk"] = (risk_df["failure_probability"] >= best_threshold).astype(int)

urgent_share = risk_df["high_risk"].mean()
backlog_count = int((risk_df["high_risk"] == 1).sum() - BASE_DAILY_CAPACITY)
backlog_count = max(0, backlog_count)
backlog_ratio = backlog_count / BASE_DAILY_CAPACITY
avg_risk_today = risk_df.loc[risk_df["inspect_today"] == 1, "failure_probability"].mean()

ops_snapshot = pd.DataFrame(
    {
        "metric": ["urgent_share", "backlog_machines", "backlog_ratio", "avg_risk_of_today_work"],
        "value": [round(urgent_share, 3), backlog_count, round(backlog_ratio, 3), round(float(avg_risk_today), 3)],
    }
)
display(ops_snapshot)


## Operational Escalation Score (Heuristic)

Pressure score is a heuristic index for monitoring load. Weights are explicitly operational assumptions, not learned from model fitting.


In [ ]:
W_URGENT_SHARE = 0.5
W_BACKLOG_RATIO = 0.3
W_RISK_LEVEL = 0.2

pressure_score = (
    W_URGENT_SHARE * urgent_share
    + W_BACKLOG_RATIO * backlog_ratio
    + W_RISK_LEVEL * float(avg_risk_today)
)

pressure_q33 = np.quantile(
    [
        W_URGENT_SHARE * urgent_share,
        W_BACKLOG_RATIO * backlog_ratio,
        W_RISK_LEVEL * float(avg_risk_today),
        pressure_score,
    ],
    0.33,
)
pressure_q66 = np.quantile(
    [
        W_URGENT_SHARE * urgent_share,
        W_BACKLOG_RATIO * backlog_ratio,
        W_RISK_LEVEL * float(avg_risk_today),
        pressure_score,
    ],
    0.66,
)

if pressure_score >= pressure_q66:
    escalation = "RED"
elif pressure_score >= pressure_q33:
    escalation = "AMBER"
else:
    escalation = "GREEN"

pressure_table = pd.DataFrame(
    {
        "metric": [
            "daily_capacity",
            "current_pressure_score",
            "pressure_q33",
            "pressure_q66",
            "escalation_status",
        ],
        "value": [BASE_DAILY_CAPACITY, round(float(pressure_score), 3), round(float(pressure_q33), 3), round(float(pressure_q66), 3), escalation],
    }
)
display(pressure_table)


## Capacity Optimization with Expected Loss

Capacity simulation minimizes total expected cost:
inspection cost + deferred expected failure loss.
This stage is the primary decision layer.


In [ ]:
def expected_cost_for_capacity(risk_table, capacity, inspect_cost=INSPECTION_COST_PER_MACHINE, failure_loss=FAILURE_LOSS_PER_MISSED_MACHINE):
    ranked = risk_table.sort_values("failure_probability", ascending=False).copy()
    ranked["rank"] = np.arange(1, len(ranked) + 1)
    ranked["inspect"] = (ranked["rank"] <= capacity).astype(int)

    inspected_count = ranked["inspect"].sum()
    deferred = ranked[ranked["inspect"] == 0]

    inspection_cost = inspected_count * inspect_cost
    deferred_expected_failure_loss = deferred["failure_probability"].sum() * failure_loss
    total_expected_cost = inspection_cost + deferred_expected_failure_loss

    return {
        "capacity": int(capacity),
        "inspection_cost": float(inspection_cost),
        "deferred_expected_failure_loss": float(deferred_expected_failure_loss),
        "total_expected_cost": float(total_expected_cost),
    }

capacity_grid = list(range(1, min(60, len(risk_df)) + 1))
cost_rows = [expected_cost_for_capacity(risk_df, c) for c in capacity_grid]
cost_df = pd.DataFrame(cost_rows)

best_cost_row = cost_df.loc[cost_df["total_expected_cost"].idxmin()]
current_row = cost_df[cost_df["capacity"] == BASE_DAILY_CAPACITY].iloc[0]

summary_cost = pd.DataFrame(
    [
        {
            "scenario": "current",
            "capacity": current_row["capacity"],
            "inspection_cost": round(current_row["inspection_cost"], 3),
            "deferred_expected_failure_loss": round(current_row["deferred_expected_failure_loss"], 3),
            "total_expected_cost": round(current_row["total_expected_cost"], 3),
        },
        {
            "scenario": "optimal",
            "capacity": best_cost_row["capacity"],
            "inspection_cost": round(best_cost_row["inspection_cost"], 3),
            "deferred_expected_failure_loss": round(best_cost_row["deferred_expected_failure_loss"], 3),
            "total_expected_cost": round(best_cost_row["total_expected_cost"], 3),
        },
    ]
)

display(summary_cost)

plt.figure(figsize=(8, 5))
plt.plot(cost_df["capacity"], cost_df["total_expected_cost"], label="total expected cost")
plt.axvline(BASE_DAILY_CAPACITY, linestyle="--", color="gray", label="current capacity")
plt.axvline(int(best_cost_row["capacity"]), linestyle=":", color="black", label="optimal capacity")
plt.xlabel("Inspection Capacity (machines/day)")
plt.ylabel("Expected Cost")
plt.title("Capacity vs Expected Cost")
plt.legend()
plt.show()

print("Current capacity:", BASE_DAILY_CAPACITY)
print("Optimal capacity under expected loss:", int(best_cost_row["capacity"]))
print("Current total expected cost:", round(float(current_row["total_expected_cost"]), 3))
print("Optimal total expected cost:", round(float(best_cost_row["total_expected_cost"]), 3))
print("Expected savings:", round(float(current_row["total_expected_cost"] - best_cost_row["total_expected_cost"]), 3))


## Optional Baseline Cost Rule

Fixed backlog penalty is kept only as a simple baseline. Expected loss remains the primary metric.


In [ ]:
FIXED_BACKLOG_COST_PER_MACHINE = 5.0

def baseline_fixed_backlog_cost(risk_table, capacity, inspect_cost=INSPECTION_COST_PER_MACHINE, backlog_cost=FIXED_BACKLOG_COST_PER_MACHINE):
    ranked = risk_table.sort_values("failure_probability", ascending=False).copy()
    ranked["rank"] = np.arange(1, len(ranked) + 1)
    ranked["inspect"] = (ranked["rank"] <= capacity).astype(int)
    backlog = int((ranked["inspect"] == 0).sum())
    total_cost = capacity * inspect_cost + backlog * backlog_cost
    return {"capacity": capacity, "backlog": backlog, "total_cost": total_cost}

baseline_demo = pd.DataFrame([baseline_fixed_backlog_cost(risk_df, c) for c in [BASE_DAILY_CAPACITY, int(best_cost_row["capacity"])]])
display(baseline_demo)


## Output Checkpoint and Stop Line

Run cells through this point and review outputs before expanding policy actions. Continue after confirming:

- validation model ranking,
- selected threshold,
- final test report and confusion matrix,
- capacity optimization table.


In [ ]:
required_outputs = {
    "validation_results_ready": "val_results" in globals(),
    "threshold_ready": "best_threshold" in globals(),
    "test_predictions_ready": "test_pred" in globals(),
    "capacity_table_ready": "summary_cost" in globals(),
}

checkpoint = pd.DataFrame(
    {"item": list(required_outputs.keys()), "ready": list(required_outputs.values())}
)
display(checkpoint)

if checkpoint["ready"].all():
    print("Stop here for review. Next stage can start after output confirmation.")
else:
    print("One or more required outputs missing. Resolve before continuing.")
